# Wine Quality Classification

**Tabular Classification Project**

## 2 · Project Overview

Classify **wine quality** (3 classes: low, medium, high) from 11 physicochemical measurements: acidity, sugar, chlorides, sulfur dioxide, density, pH, sulphates, and alcohol. Synthetic red wine data with ~1,600 rows. Quality is notoriously hard to predict from chemistry alone.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given 11 physicochemical properties of red wine, classify quality as low (0), medium (1), or high (2).

## 5 · Why This Project Matters

- **Wine quality prediction** is a well-known ML benchmark problem.
- Quality ratings are subjective — chemistry alone cannot fully predict them.
- This teaches working with **ordinal targets** binned into classes.
- Alcohol content is the strongest predictor — simple features often dominate.
- The dataset is moderately difficult — most models achieve ~70-80% accuracy.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~1,600 |
| Features | 11 (fixed_acidity, volatile_acidity, citric_acid, residual_sugar, chlorides, free_sulfur_dioxide, total_sulfur_dioxide, density, pH, sulphates, alcohol) |
| Target | `quality` (3 classes: 0=low, 1=medium, 2=high) |
| Class balance | Medium dominates (~70%), low (~15%), high (~15%) |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: UCI Wine Quality dataset (Cortez et al., 2009).
**License**: Public domain / educational.
**Note**: Synthetic reproduction of wine physicochemical measurements.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "quality"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Wine Quality Classification


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 1599

fixed_acidity = np.random.normal(8.3, 1.7, n).clip(4, 16).round(1)
volatile_acidity = np.random.normal(0.53, 0.18, n).clip(0.1, 1.6).round(2)
citric_acid = np.random.beta(2, 3, n).clip(0, 1).round(2)
residual_sugar = np.random.lognormal(0.5, 0.5, n).clip(0.9, 15.5).round(1)
chlorides = np.random.lognormal(-2.5, 0.4, n).clip(0.01, 0.6).round(3)
free_so2 = np.random.lognormal(2.5, 0.6, n).clip(1, 72).astype(int)
total_so2 = (free_so2 * np.random.uniform(1.5, 5, n)).clip(6, 289).astype(int)
density = np.random.normal(0.9967, 0.002, n).clip(0.99, 1.004).round(4)
ph = np.random.normal(3.31, 0.15, n).clip(2.7, 4.0).round(2)
sulphates = np.random.lognormal(-0.5, 0.3, n).clip(0.3, 2.0).round(2)
alcohol = np.random.normal(10.4, 1.1, n).clip(8, 15).round(1)

# Quality based primarily on alcohol and volatile acidity
score = (
    0.4 * (alcohol - 9)
    - 1.5 * (volatile_acidity - 0.4)
    + 0.5 * sulphates
    + 0.2 * citric_acid
    + np.random.normal(0, 0.8, n)
)
quality = np.digitize(score, bins=[-0.5, 1.5]) # 0=low, 1=medium, 2=high

df = pd.DataFrame({
    'fixed_acidity': fixed_acidity, 'volatile_acidity': volatile_acidity,
    'citric_acid': citric_acid, 'residual_sugar': residual_sugar,
    'chlorides': chlorides, 'free_sulfur_dioxide': free_so2,
    'total_sulfur_dioxide': total_so2, 'density': density, 'pH': ph,
    'sulphates': sulphates, 'alcohol': alcohol, 'quality': quality,
})

# Ensure all classes well-represented
for cls in range(3):
    if (df['quality'] == cls).sum() < 50:
        idx = np.random.choice(df.index, 80, replace=False)
        df.loc[idx, 'quality'] = cls

print(f"Dataset shape: {df.shape}")
print(f"Quality distribution:\n{df['quality'].value_counts().sort_index()}")
df.head()

Dataset shape: (1599, 12)
Quality distribution:
quality
0     153
1    1095
2     351
Name: count, dtype: int64


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
0,9.1,0.40,0.26,1.5,0.051,5,15,0.9932,3.33,0.50,10.7,1
1,8.1,0.50,0.27,0.9,0.088,16,57,0.9984,3.27,0.76,9.5,1
2,9.4,0.78,0.30,1.5,0.039,29,92,0.9942,3.36,0.60,9.5,1
3,10.9,0.41,0.47,2.7,0.099,19,90,0.9949,3.16,0.82,11.6,2
4,7.9,0.39,0.45,1.9,0.079,20,46,0.9972,3.24,0.46,11.1,1


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (1599, 12)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
quality
1    1095
2     351
0     153
Name: count, dtype: int64

Dtypes:
fixed_acidity           float64
volatile_acidity        float64
citric_acid             float64
residual_sugar          float64
chlorides               float64
free_sulfur_dioxide       int64
total_sulfur_dioxide      int64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

Target 'quality' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for i, col in enumerate(num_cols[:11]):
    ax = axes[i // 4, i % 4]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col, fontsize=9)
if len(num_cols) < 12:
    axes[2, 3].set_visible(False)
plt.suptitle("Features by Wine Quality", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

corr_target = df[num_cols].corrwith(df[TARGET]).sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
corr_target.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title("Feature Correlation with Quality")
plt.tight_layout()
plt.show()
print(f"Top correlated: alcohol ({corr_target['alcohol']:.3f}), volatile_acidity ({corr_target['volatile_acidity']:.3f})")

Top correlated: alcohol (0.393), volatile_acidity (-0.211)


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['coral', 'steelblue', 'gold'], edgecolor='black')
axes[0].set_title("Wine Quality Distribution")
axes[0].set_xticklabels(['Low (0)', 'Medium (1)', 'High (2)'], rotation=0)
df[TARGET].value_counts().sort_index().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts().sort_index()}")

Class distribution:
quality
0     153
1    1095
2     351
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().sort_index().to_dict()}")

Train: (1279, 11), Test: (320, 11)
Train target dist: {0: 122, 1: 876, 2: 281}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
X_train = X_train.copy(); X_test = X_test.copy()
X_train['free_so2_ratio'] = X_train['free_sulfur_dioxide'] / (X_train['total_sulfur_dioxide'] + 1)
X_test['free_so2_ratio'] = X_test['free_sulfur_dioxide'] / (X_test['total_sulfur_dioxide'] + 1)
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean; X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (12): ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'free_so2_ratio']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
n_classes = len(np.unique(y_train))
if n_classes == 2:
    y_prob_bl = baseline.predict_proba(X_test)[:, 1]
else:
    y_prob_bl = None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.6969
Precision: 0.7037
Recall   : 0.6969
F1       : 0.6307


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
NearestCentroid                0.425000           0.574818  0.629362  0.424899   0.598571  0.425000    0.016845
BaggingClassifier              0.650000           0.409079  0.569014  0.613217   0.623664  0.650000    0.088506
QuadraticDiscriminantAnalysis  0.668750           0.406230  0.601905  0.625030   0.608510  0.668750    0.015999
LabelPropagation               0.571875           0.405688  0.559466  0.572530   0.573728  0.571875    0.062286
LabelSpreading                 0.571875           0.405688  0.562968  0.572530   0.573728  0.571875    0.084960
ExtraTreeClassifier            0.546875           0.405004  0.522555  0.550860   0.555222  0.546875    0.019149
GaussianNB                     0.684375           0.400881  0.620276  

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="macro_f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.6250
F1       : 0.5842


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test).ravel()
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.6531  F1: 0.5993


LightGBM  Acc: 0.6375  F1: 0.6025


XGBoost   Acc: 0.6312  F1: 0.5883


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.6969  0.6307     0.7037  0.6969
LightGBM               0.6375  0.6025     0.5995  0.6375
CatBoost               0.6531  0.5993     0.6079  0.6531
XGBoost                0.6312  0.5883     0.5833  0.6312
FLAML                  0.6250  0.5842     0.5706  0.6250

Top 3: ['Logistic Regression', 'LightGBM', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  Logistic Regression
              precision    recall  f1-score   support

           0       1.00      0.03      0.06        31
           1       0.71      0.94      0.81       219
           2       0.55      0.23      0.32        70

    accuracy                           0.70       320
   macro avg       0.75      0.40      0.40       320
weighted avg       0.70      0.70      0.63       320


  LightGBM
              precision    recall  f1-score   support

           0       0.40      0.06      0.11        31
           1       0.70      0.83      0.76       219
           2       0.37      0.30      0.33        70

    accuracy                           0.64       320
   macro avg       0.49      0.40      0.40       320
weighted avg       0.60      0.64      0.60       320


  CatBoost
              precision    recall  f1-score   support

           0       0.50      0.03      0.06        31
           1       0.70      0.88      0

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: Logistic Regression
Error rate: 0.3031 (97 / 320)

Errors by true class:
      errors  total  error_rate
true                           
0         30     31    0.967742
1         13    219    0.059361
2         54     70    0.771429


## 25 · Interpretation and Business Insight

- **Alcohol** is the strongest predictor of quality — higher alcohol correlates with higher quality.
- **Volatile acidity** negatively correlates — too much vinegar-like taste reduces quality.
- **Sulphates** (antimicrobial) and **citric acid** (freshness) positively affect quality.
- **Residual sugar** and **pH** have weak predictive power.
- Quality is inherently subjective — chemistry alone has a ceiling on predictability.

## 26 · Limitations

1. Subjective quality ratings — different tasters disagree.
2. Only red wine — white wine has different quality drivers.
3. 3-class binning loses continuous quality information.
4. No sensory descriptors (aroma, taste notes).
5. No vineyard, grape variety, or vintage information.

## 27 · How to Improve This Project

1. Use original 1-10 quality scores for regression.
2. Combine red and white wine datasets.
3. Add sensory panel descriptors.
4. Include vineyard metadata (climate, soil, altitude).
5. Try ordinal regression (quality is ordered).

## 28 · Production Considerations

- Wine quality control during production.
- Blending optimization using chemical targets.
- Consumer recommendation based on chemical profile.
- Must complement, not replace, expert tasting panels.

## 29 · Common Mistakes

1. Ignoring alcohol as the dominant driver.
2. Using accuracy on imbalanced 3-class data.
3. Not trying ordinal regression for ordered quality levels.
4. Over-engineering when the problem has an inherent ceiling.
5. Treating quality as objective when it's subjective.

## 30 · Mini Challenge / Exercises

1. Build a model with only alcohol and volatile_acidity — what accuracy?
2. Compare 3-class vs original 1-10 regression — which is easier?
3. Which quality level is hardest to predict? Why?
4. Try ordinal regression — does respecting order help?

## 31 · Final Summary / Key Takeaways

1. Wine quality prediction is bounded by subjective ratings.
2. Alcohol content is the strongest chemical predictor.
3. Volatile acidity hurts quality — vinegar-like taste is undesirable.
4. Chemistry alone cannot perfectly predict quality.
5. Real wine quality assessment combines chemistry with expert tasting.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.6531,
    "f1": 0.5993,
    "precision": 0.6079,
    "recall": 0.6531
  },
  "LightGBM": {
    "accuracy": 0.6375,
    "f1": 0.6025,
    "precision": 0.5995,
    "recall": 0.6375
  },
  "XGBoost": {
    "accuracy": 0.6312,
    "f1": 0.5883,
    "precision": 0.5833,
    "recall": 0.6312
  },
  "Logistic Regression": {
    "accuracy": 0.6969,
    "f1": 0.6307,
    "precision": 0.7037,
    "recall": 0.6969
  },
  "FLAML": {
    "accuracy": 0.625,
    "f1": 0.5842,
    "precision": 0.5706,
    "recall": 0.625
  }
}
